# 02. Retrieval Baseline
BM25 / Embedding 3종(OpenAI·BGE-M3·ko-sroberta) / Hybrid RRF
평가: Recall@5 · MRR · nDCG@5
팀원 A 담당

In [ ]:
import pandas as pd
import numpy as np
import json
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# 데이터 로드
corpus   = pd.read_parquet('../data/insk_corpus.parquet')
analyses = pd.read_parquet('../data/article_analyses.parquet')
emb_df   = pd.read_parquet('../data/article_embeddings.parquet')
qa_list  = [json.loads(l) for l in open('../data/human_qa_benchmark_v1.jsonl', encoding='utf-8')]

# 병합
df = corpus.merge(analyses, on='article_id', how='left')

# article_id 순서 고정 (인덱스 ↔ article_id 매핑용)
df = df.reset_index(drop=True)
article_ids = df['article_id'].tolist()          # 인덱스 → article_id
id_to_idx   = {aid: i for i, aid in enumerate(article_ids)}  # article_id → 인덱스

print(f'기사 수: {len(df)}')
print(f'QA 수:   {len(qa_list)}')

## 공통 — 텍스트 조합 & 평가 함수

In [ ]:
# BM25/Embedding에 넣을 텍스트 조합 (body 없으므로 아래 필드만 사용)
def build_doc_text(row):
    title   = str(row['title'])   if pd.notna(row['title'])   else ''
    summary = str(row['summary']) if pd.notna(row['summary']) else ''
    insight = str(row['insight']) if pd.notna(row['insight']) else ''
    tags    = ' '.join(json.loads(row['tags'])) if pd.notna(row['tags']) else ''
    return f"{title} {summary} {insight} {tags}"

doc_texts = df.apply(build_doc_text, axis=1).tolist()
print('문서 텍스트 예시:')
print(doc_texts[0][:200], '...')

In [ ]:
# ── 평가 지표 함수 ──

def recall_at_k(retrieved_ids, gold_ids, k=5):
    """상위 k개 중 gold가 몇 개 포함되는지"""
    if not gold_ids:
        return None
    hits = len(set(retrieved_ids[:k]) & set(gold_ids))
    return hits / len(gold_ids)

def mrr(retrieved_ids, gold_ids):
    """첫 번째 정답의 역순위"""
    if not gold_ids:
        return None
    for rank, rid in enumerate(retrieved_ids, 1):
        if rid in gold_ids:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_ids, gold_ids, k=5):
    """nDCG@k (관련 문서 모두 동일 relevance=1)"""
    if not gold_ids:
        return None
    dcg = sum(
        1.0 / np.log2(rank + 1)
        for rank, rid in enumerate(retrieved_ids[:k], 1)
        if rid in gold_ids
    )
    ideal_hits = min(len(gold_ids), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(retrieval_fn, qa_list, k=5):
    """
    retrieval_fn(question) → article_id 리스트 (순위 순)
    Strict / Trend / Negative 유형별 + 전체 평균 반환
    """
    results = {'Strict': [], 'Trend': [], 'Negative': []}

    for q in qa_list:
        retrieved = retrieval_fn(q['question'])
        gold      = q.get('gold_articles', [])
        qtype     = q['type']

        if qtype == 'Negative':
            # Negative: gold 없음 → 아무것도 안 가져오면 좋음 (hallucination 평가)
            hit = any(rid in retrieved[:k] for rid in retrieved[:k])  # 항상 True
            abstained = len(retrieved) == 0
            results['Negative'].append({'abstained': abstained})
        else:
            r = recall_at_k(retrieved, gold, k)
            m = mrr(retrieved, gold)
            n = ndcg_at_k(retrieved, gold, k)
            results[qtype].append({'recall': r, 'mrr': m, 'ndcg': n,
                                   'question': q['question'], 'retrieved': retrieved[:k], 'gold': gold})

    # 평균 계산
    summary = {}
    for qtype in ['Strict', 'Trend']:
        vals = results[qtype]
        if vals:
            summary[qtype] = {
                'Recall@5': np.mean([v['recall'] for v in vals if v['recall'] is not None]),
                'MRR':      np.mean([v['mrr']    for v in vals if v['mrr']    is not None]),
                'nDCG@5':   np.mean([v['ndcg']   for v in vals if v['ndcg']   is not None]),
            }
    all_vals = results['Strict'] + results['Trend']
    summary['전체'] = {
        'Recall@5': np.mean([v['recall'] for v in all_vals if v['recall'] is not None]),
        'MRR':      np.mean([v['mrr']    for v in all_vals if v['mrr']    is not None]),
        'nDCG@5':   np.mean([v['ndcg']   for v in all_vals if v['ndcg']   is not None]),
    }
    return summary, results

def print_summary(name, summary):
    print(f'\n{'='*50}')
    print(f'{name}')
    print(f'{'─'*50}')
    for qtype, metrics in summary.items():
        print(f'  {qtype:8s}  Recall@5={metrics["Recall@5"]:.3f}  MRR={metrics["MRR"]:.3f}  nDCG@5={metrics["nDCG@5"]:.3f}')

print('평가 함수 정의 완료')

## 실험 1-A. BM25

In [ ]:
from rank_bm25 import BM25Okapi

# 토크나이징 (공백 분리 — 형태소 분석기 없어도 한국어 어느 정도 동작)
tokenized_corpus = [text.split() for text in doc_texts]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(question, top_k=10):
    tokens  = question.split()
    scores  = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [article_ids[i] for i in top_idx]

# 테스트
sample_q = qa_list[0]['question']
print(f'질문: {sample_q}')
print(f'BM25 top-5: {bm25_retrieve(sample_q, top_k=5)}')
print(f'정답:       {qa_list[0]["gold_articles"]}')

In [ ]:
bm25_summary, bm25_results = evaluate(lambda q: bm25_retrieve(q, top_k=10), qa_list)
print_summary('BM25 only', bm25_summary)

## 실험 1-B. Embedding (OpenAI 1536d) — 이미 계산된 벡터 활용

In [ ]:
# OpenAI 임베딩 로드 — null 3건(article_id 166·167·168) 안전 처리
def parse_embedding(s):
    if pd.isna(s): return None
    parsed = json.loads(s)
    if parsed is None or not isinstance(parsed, list): return None
    return np.array(parsed, dtype=np.float32)

emb_df["embedding"] = emb_df["embedding_json"].apply(parse_embedding)
emb_merged = df[["article_id"]].merge(emb_df[["article_id","embedding"]], on="article_id", how="left")

# null 제외한 유효 임베딩만 사용
valid_mask = emb_merged["embedding"].notna()
valid_article_ids_emb = df[valid_mask.values]["article_id"].tolist()
openai_vectors = np.vstack(emb_merged[valid_mask]["embedding"].values).astype(np.float32)
print(f"유효 임베딩: {len(openai_vectors)}건 (null {(~valid_mask).sum()}건 제외)")

faiss.normalize_L2(openai_vectors)
index_openai = faiss.IndexFlatIP(1536)
index_openai.add(openai_vectors)
print(f"OpenAI FAISS 인덱스 구축 완료: {index_openai.ntotal}개")


In [ ]:
# API 키 로드 — .env 파일에서만 읽음 (코드에 직접 입력 금지)
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()  # .env 파일 자동 로드
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️  .env 파일에 OPENAI_API_KEY가 없습니다")
    print("   사용 전 반드시 Park에게 허락 받고 .env에 입력하세요")
else:
    client = OpenAI(api_key=api_key)
    print("✅ OpenAI 클라이언트 준비 완료")


In [ ]:
openai_summary, openai_results = evaluate(lambda q: openai_retrieve(q, top_k=10), qa_list)
print_summary('OpenAI Embedding (text-embedding-3-small)', openai_summary)

## 실험 1-C. Embedding (BGE-M3)

In [ ]:
# ⚠️ 모델 다운로드 약 2GB, 최초 1회만 다운로드됨
print('BGE-M3 모델 로드 중... (처음이면 다운로드 수분 소요)')
model_bge = SentenceTransformer('BAAI/bge-m3')
print('로드 완료')

# 전체 corpus 인코딩
bge_vectors = model_bge.encode(doc_texts, batch_size=32, show_progress_bar=True,
                                normalize_embeddings=True).astype(np.float32)
print(f'BGE-M3 벡터 shape: {bge_vectors.shape}')

dim_bge = bge_vectors.shape[1]
index_bge = faiss.IndexFlatIP(dim_bge)
index_bge.add(bge_vectors)
print(f'BGE-M3 FAISS 인덱스 구축 완료')

In [ ]:
def bge_retrieve(question, top_k=10):
    q_vec = model_bge.encode([question], normalize_embeddings=True).astype(np.float32)
    _, indices = index_bge.search(q_vec, top_k)
    return [article_ids[i] for i in indices[0]]

bge_summary, bge_results = evaluate(lambda q: bge_retrieve(q, top_k=10), qa_list)
print_summary('BGE-M3', bge_summary)

## 실험 1-D. Embedding (ko-sroberta)

In [ ]:
print('ko-sroberta 모델 로드 중...')
model_ko = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('로드 완료')

ko_vectors = model_ko.encode(doc_texts, batch_size=32, show_progress_bar=True,
                              normalize_embeddings=True).astype(np.float32)
print(f'ko-sroberta 벡터 shape: {ko_vectors.shape}')

dim_ko = ko_vectors.shape[1]
index_ko = faiss.IndexFlatIP(dim_ko)
index_ko.add(ko_vectors)
print('ko-sroberta FAISS 인덱스 구축 완료')

In [ ]:
def ko_retrieve(question, top_k=10):
    q_vec = model_ko.encode([question], normalize_embeddings=True).astype(np.float32)
    _, indices = index_ko.search(q_vec, top_k)
    return [article_ids[i] for i in indices[0]]

ko_summary, ko_results = evaluate(lambda q: ko_retrieve(q, top_k=10), qa_list)
print_summary('ko-sroberta', ko_summary)

## 실험 1-E. Hybrid RRF (BM25 + OpenAI Embedding)

In [ ]:
def reciprocal_rank_fusion(rankings_list, rrf_k=60):
    """여러 랭킹 리스트를 RRF로 합산 → article_id 순위 리스트 반환"""
    scores = {}
    for rankings in rankings_list:
        for rank, article_id in enumerate(rankings):
            scores[article_id] = scores.get(article_id, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.keys(), key=lambda x: -scores[x])

def hybrid_retrieve(question, top_k=10):
    bm25_top   = bm25_retrieve(question, top_k=top_k)
    openai_top = openai_retrieve(question, top_k=top_k)
    return reciprocal_rank_fusion([bm25_top, openai_top])[:top_k]

# 테스트
print(f'질문: {sample_q}')
print(f'Hybrid top-5: {hybrid_retrieve(sample_q, top_k=5)}')
print(f'정답:         {qa_list[0]["gold_articles"]}')

In [ ]:
hybrid_summary, hybrid_results = evaluate(lambda q: hybrid_retrieve(q, top_k=10), qa_list)
print_summary('Hybrid RRF (BM25 + OpenAI)', hybrid_summary)

## 최종 비교 표

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

models = {
    'BM25':         bm25_summary,
    'OpenAI Emb':   openai_summary,
    'BGE-M3':       bge_summary,
    'ko-sroberta':  ko_summary,
    'Hybrid RRF':   hybrid_summary,
}

rows = []
for model_name, summary in models.items():
    for qtype in ['Strict', 'Trend', '전체']:
        if qtype in summary:
            rows.append({
                'Model': model_name,
                'Type':  qtype,
                **summary[qtype]
            })

result_df = pd.DataFrame(rows)
pivot = result_df.pivot(index='Model', columns='Type', values='Recall@5')

print('=== Recall@5 비교 (행=모델, 열=QA유형) ===')
print(result_df[result_df['Type']=='전체'][['Model','Recall@5','MRR','nDCG@5']].to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['Recall@5', 'MRR', 'nDCG@5']
colors  = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']

for ax, metric in zip(axes, metrics):
    vals = result_df[result_df['Type']=='전체'].set_index('Model')[metric]
    bars = ax.bar(vals.index, vals.values, color=colors)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Retrieval 방식별 성능 비교 (전체 QA 21개)', fontsize=14)
plt.tight_layout()
plt.show()

## Query 특성별 분석

In [ ]:
print('=== Query 유형별 특성 분석 ===')
print()
print('Strict QA (정답 1-2개 명확) — BM25 강점 예상')
print('Trend  QA (정답 다수)       — Embedding 강점 예상')
print()

for model_name, summary in models.items():
    strict_r = summary.get('Strict', {}).get('Recall@5', 0)
    trend_r  = summary.get('Trend',  {}).get('Recall@5', 0)
    diff     = trend_r - strict_r
    label    = '(Trend 강)' if diff > 0.05 else ('(Strict 강)' if diff < -0.05 else '(균형)')
    print(f'  {model_name:12s}  Strict={strict_r:.3f}  Trend={trend_r:.3f}  {label}')

## 팀원 B에게 전달할 Retrieval Top-10 저장

In [ ]:
# 팀원 B의 Hard Negative Mining에 필요한 파일
# Hybrid RRF top-10 결과를 저장

retrieval_outputs = []
for q in qa_list:
    if q['type'] == 'Negative':
        continue
    top10 = hybrid_retrieve(q['question'], top_k=10)
    retrieval_outputs.append({
        'id':           q['id'],
        'question':     q['question'],
        'type':         q['type'],
        'gold_articles': q['gold_articles'],
        'hybrid_top10': top10,
        # hard negatives = top10 중 gold 제외한 것
        'hard_negatives': [x for x in top10 if x not in q['gold_articles']]
    })

out_path = '../data/retrieval_top10_for_reranker.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for row in retrieval_outputs:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'저장 완료: {out_path}')
print(f'총 {len(retrieval_outputs)}개 QA의 retrieval top-10 + hard negatives 저장')
print('→ 팀원 B에게 이 파일 공유하면 Hard Negative Mining 바로 시작 가능')

## 실험 결과 요약

In [ ]:
print('='*55)
print('최종 결과 요약 (전체 QA 기준)')
print('='*55)
print(f'{"방식":<14} {"Recall@5":>9} {"MRR":>8} {"nDCG@5":>8}')
print('-'*55)
for model_name, summary in models.items():
    m = summary.get('전체', {})
    print(f'{model_name:<14} {m.get("Recall@5",0):>9.3f} {m.get("MRR",0):>8.3f} {m.get("nDCG@5",0):>8.3f}')
print('='*55)
print()
print('다음 단계:')
print('  - data/retrieval_top10_for_reranker.jsonl → 팀원 B')
print('  - Hybrid RRF가 최고 성능이면 이 결과를 baseline으로 사용')